# SimpleDyG – bảng và biểu đồ theo kiểu paper (chỉ CPU)

Notebook này **không huấn luyện**. Nó đọc kết quả đã có và vẽ lại các bảng/hình của paper:

| Paper | Ở đây |
|---|---|
| Table 1 – thống kê dataset | `table1_datasets` (+ số truy vấn split, chuỗi bị cắt) |
| Table 2 – so với baseline | bảng (đậm = tốt nhất, gạch chân = thứ hai) + biểu đồ cột, ◇ = số paper, nét đứt = baseline không huấn luyện |
| Table 4 – temporal alignment | bảng + Δ so cặp theo seed với CI 95% + biểu đồ |
| Table 5 – thời gian/epoch | `table5_time` (T4, gồm cả validation mỗi epoch) |
| Figure A.1 – phân bố tương tác theo time step | `fig_A1_temporal_pattern` |
| Figure 4(c) – hidden dimension | chỉ cho TGAT/GraphMixer (lưới chọn dim trên validation) |
| (thêm) chẩn đoán | đường validation theo epoch, so cặp từng seed, kiểm tra suy biến dự đoán |

Figure 3 (dự đoán nhiều bước) và sweep layer/head của Figure 4 không được chạy nên không vẽ.

**Input (Add Input trên Kaggle, chọn một hoặc nhiều):**
- Output của notebook 100 lượt (thư mục có `results_per_seed.csv` và `runs/`), **hoặc** `simple_dyg_result.zip`, **hoặc** `SimpleDyG_Previous_Results.zip` (1,3 MB, đủ cho mọi hình).
- Tùy chọn: Output của `SimpleDyG_Completion_2GPU.ipynb` (`uci_permute`, `ml_corrected`) – được gộp tự động.
- Tùy chọn: `SimpleDyG_Kaggle_2GPU_Inputs.zip` – cần cho Table 1 (số time step), Figure A.1 và các dòng baseline không huấn luyện.

Không cần GPU. Run All. Kết quả: `simpledyg_paper_figures.zip` (PNG, CSV, `paper_tables.html`).


In [ ]:
NDCG_METRIC = "ndcg5"        # "ndcg5" = NDCG chuẩn (khử trùng lặp); "ndcg5_author" = cách tính trong mã/paper
INPUT_DIR = ""               # trống = /kaggle/input; khi chạy máy cá nhân: thư mục chứa các ZIP/Output
EXTRA_RESULT_ROOTS = []      # thêm thư mục kết quả thủ công nếu cần (mỗi thư mục có results_per_seed.csv)
AUTHOR_ROOT_OVERRIDE = ""    # thư mục SimpleDyG có resources/ và all_data/ (nếu không dùng ZIP input)


## 1. Môi trường

In [ ]:
import os, sys, subprocess, json, zipfile, shutil, types
from pathlib import Path
try:
    import matplotlib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "matplotlib>=3.7,<4"], check=True)
import pandas as pd
INPUT = Path(INPUT_DIR) if INPUT_DIR else Path("/kaggle/input")
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "simpledyg_figures_work"
WORK.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(WORK / "mpl_cache"))
print("Input:", INPUT, "| Work:", WORK)


## 2. Tìm kết quả và dữ liệu

In [ ]:
# Tìm kết quả trong thư mục và trong ZIP; chỉ giải nén file nhỏ cần cho bảng/hình (bỏ qua *.pt).
EXTRACT = WORK / "extracted_results"
KEEP_TOP = {"results_per_seed.csv", "data_audit.json"}
KEEP_RUN = ("/history.json", "/validation_search.json", "/test_predictions.jsonl")

def safe_extract(archive, member, target):
    dest = (target / member["rel"]).resolve()
    if not dest.is_relative_to(target.resolve()):
        raise ValueError("ZIP path escapes target: " + member["rel"])
    dest.parent.mkdir(parents=True, exist_ok=True)
    with archive.open(member["name"]) as src, open(dest, "wb") as out:
        shutil.copyfileobj(src, out)

roots = [Path(p) for p in EXTRA_RESULT_ROOTS]
if INPUT.exists():
    roots += sorted(p.parent for p in INPUT.rglob("results_per_seed.csv") if p.stat().st_size > 1)
    for z in sorted(INPUT.rglob("*.zip")):
        try:
            archive = zipfile.ZipFile(z)
        except zipfile.BadZipFile:
            continue
        with archive:
            names = archive.namelist()
            prefixes = sorted({n[:-len("results_per_seed.csv")] for n in names if n.endswith("results_per_seed.csv")})
            for k, prefix in enumerate(prefixes):
                target = EXTRACT / f"{z.stem}_{k}"
                members = []
                for n in names:
                    if n.startswith(prefix) and not n.endswith("/"):
                        rel = n[len(prefix):]
                        if rel in KEEP_TOP or (rel.startswith("runs/") and rel.endswith(KEEP_RUN)):
                            members.append(dict(name=n, rel=rel))
                for m in members:
                    safe_extract(archive, m, target)
                print(f"ZIP {z.name}: {len(members)} file kết quả -> {target}")
                roots.append(target)
# Bỏ thư mục trùng; kết quả trùng run_id được gộp trong paper_figures.load_results.
seen, RESULT_ROOTS = set(), []
for r in roots:
    key = r.resolve()
    if key not in seen and (r / "results_per_seed.csv").exists():
        seen.add(key)
        RESULT_ROOTS.append(r)
if not RESULT_ROOTS:
    raise FileNotFoundError("Không tìm thấy results_per_seed.csv. Add Input Output 100 lượt, simple_dyg_result.zip "
                            "hoặc SimpleDyG_Previous_Results.zip.")
for r in RESULT_ROOTS:
    n = len(pd.read_csv(r / "results_per_seed.csv"))
    print(f"{n:4d} run | {r}")

# Dữ liệu gốc cho Table 1 / Figure A.1 / baseline không huấn luyện (không bắt buộc).
AUTHOR_ROOT = Path(AUTHOR_ROOT_OVERRIDE) if AUTHOR_ROOT_OVERRIDE else None
if AUTHOR_ROOT is None and INPUT.exists():
    for p in INPUT.rglob("train.link_prediction"):
        if p.parent.name == "12" and p.parent.parent.name == "UCI_13":
            AUTHOR_ROOT = p.parents[3]
            break
if AUTHOR_ROOT is None and INPUT.exists():
    for z in sorted(INPUT.rglob("*.zip")):
        try:
            archive = zipfile.ZipFile(z)
        except zipfile.BadZipFile:
            continue
        with archive:
            names = archive.namelist()
            anchor = [n for n in names if n.endswith("resources/UCI_13/12/train.link_prediction")]
            if not anchor:
                continue
            prefix = anchor[0][:-len("resources/UCI_13/12/train.link_prediction")]
            target = WORK / "author_data"
            for n in names:
                rel = n[len(prefix):] if n.startswith(prefix) else None
                if rel and not n.endswith("/") and (rel.startswith("resources/") or (rel.startswith("all_data/") and rel.endswith(".csv"))):
                    safe_extract(archive, dict(name=n, rel=rel), target)
            AUTHOR_ROOT = target
            break
print("Dữ liệu gốc:", AUTHOR_ROOT or "không có – bỏ qua Figure A.1, số time step và baseline không huấn luyện")


## 3. Dựng bảng và biểu đồ

In [ ]:
PAPER_FIGURES_SOURCE = '"""Paper-style tables and figures for the SimpleDyG reproduction (CPU only, no torch).\n\nMirrors what the paper reports where our runs allow it:\n  Table 1  dataset statistics                -> table1_datasets\n  Table 2  link prediction vs baselines      -> table2_performance + fig_table2_bars.png\n  Table 4  temporal alignment ablation       -> table4_temporal + fig_table4_ablation.png\n  Table 5  training time per epoch           -> table5_time\n  Fig. A.1 temporal pattern of each dataset  -> fig_A1_temporal_pattern.png\n  Fig. 4c  hidden dimension (baselines\' validation search) -> fig_baseline_dims.png\nExtra diagnostics: per-seed paired plot, validation curves, prediction-collapse plot.\nFigure 3 (multi-step) and the layer/head sweeps of Figure 4 were not run and are not drawn.\n\nEntry point: render_all(result_roots, author_root, out_dir) -> dict of DataFrames/paths.\n"""\nimport collections\nimport html\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nDATASETS = [\'UCI_13\', \'ML_10M_13\']\nLABEL = {\'UCI_13\': \'UCI\', \'ML_10M_13\': \'ML-10M\'}\nRAW = {\'UCI_13\': (\'uci\', 12), \'ML_10M_13\': (\'movie\', 12)}\nDOMAIN = {\'UCI_13\': \'Social\', \'ML_10M_13\': \'Rating\'}\nPAPER_T1 = {\'UCI_13\': (1781, 16743), \'ML_10M_13\': (15841, 48561)}\n# Table 2 (NDCG@5 mean, std, Jaccard mean, std) for UCI and ML-10M.\nPAPER_T2 = {\n    \'DySAT\': {\'UCI_13\': (.010, .003, .010, .001), \'ML_10M_13\': (.058, .073, .050, .068)},\n    \'EvolveGCN\': {\'UCI_13\': (.064, .045, .032, .026), \'ML_10M_13\': (.097, .071, .092, .067)},\n    \'DyRep\': {\'UCI_13\': (.011, .018, .010, .005), \'ML_10M_13\': (.064, .036, .038, .001)},\n    \'JODIE\': {\'UCI_13\': (.022, .023, .012, .009), \'ML_10M_13\': (.059, .016, .020, .004)},\n    \'TGAT\': {\'UCI_13\': (.061, .007, .020, .002), \'ML_10M_13\': (.066, .035, .021, .007)},\n    \'TGN\': {\'UCI_13\': (.041, .017, .011, .003), \'ML_10M_13\': (.071, .029, .023, .001)},\n    \'TREND\': {\'UCI_13\': (.067, .010, .039, .020), \'ML_10M_13\': (.079, .028, .024, .003)},\n    \'GraphMixer\': {\'UCI_13\': (.104, .013, .042, .005), \'ML_10M_13\': (.081, .033, .043, .022)},\n    \'SimpleDyG\': {\'UCI_13\': (.104, .010, .092, .014), \'ML_10M_13\': (.138, .009, .131, .008)},\n}\nPAPER_T4 = {\n    \'full\': PAPER_T2[\'SimpleDyG\'],\n    \'same_time\': {\'UCI_13\': (.090, .013, .083, .012), \'ML_10M_13\': (.147, .001, .139, .001)},\n    \'no_time\': {\'UCI_13\': (.111, .015, .091, .014), \'ML_10M_13\': (.117, .062, .111, .059)},\n}\nPAPER_T5 = {\'DySAT\': 12.24, \'EvolveGCN\': 11.89, \'DyRep\': 6.4, \'JODIE\': 6.25, \'TGAT\': 18.54,\n            \'TGN\': 8.05, \'TREND\': 7.45, \'GraphMixer\': 6.89, \'SimpleDyG\': 6.21}\nVARIANT_ORDER = [\'full\', \'same_time\', \'no_time\', \'per_ego_permute\']\nVARIANT_LABEL = {\'full\': \'SimpleDyG (full)\', \'same_time\': \'same time\', \'no_time\': \'no time\',\n                 \'per_ego_permute\': \'per-ego permuted time\'}\n# Validated categorical slots (blue, orange, aqua, violet); color follows the variant.\nCOLOR = {\'full\': \'#2a78d6\', \'same_time\': \'#eb6834\', \'no_time\': \'#1baf7a\', \'per_ego_permute\': \'#6250d6\',\n         \'TGAT\': \'#eb6834\', \'GraphMixer\': \'#1baf7a\', \'SimpleDyG\': \'#2a78d6\'}\nINK, INK2, GRID, NEUTRAL = \'#0b0b0b\', \'#52514e\', \'#e6e5e1\', \'#8f8d88\'\nT975 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262,\n        10: 2.228, 11: 2.201, 12: 2.179, 13: 2.160, 14: 2.145}\n\n\n# ----------------------------------------------------------------- loading\ndef load_results(result_roots):\n    frames, runs = [], {}\n    for root in map(Path, result_roots):\n        table = root / \'results_per_seed.csv\'\n        if table.exists() and table.stat().st_size > 1:\n            frames.append(pd.read_csv(table))\n        if (root / \'runs\').is_dir():\n            for d in (root / \'runs\').iterdir():\n                if d.is_dir():\n                    runs.setdefault(d.name, d)\n    if not frames:\n        raise FileNotFoundError(\'No results_per_seed.csv in \' + str(list(result_roots)))\n    df = pd.concat(frames, ignore_index=True).drop_duplicates(\'run_id\', keep=\'last\')\n    df[\'seed\'] = df.seed.astype(int)\n    return df, runs\n\n\ndef load_audit(result_roots):\n    audit = {}\n    for root in map(Path, result_roots):\n        p = root / \'data_audit.json\'\n        if p.exists():\n            for k, v in json.loads(p.read_text(encoding=\'utf-8\')).items():\n                audit.setdefault(k, v)\n    return audit\n\n\ndef _fmt(m, s=None, digits=3):\n    if m is None or (isinstance(m, float) and math.isnan(m)):\n        return \'–\'\n    text = f\'{m:.{digits}f}\'.replace(\'0.\', \'.\', 1) if abs(m) < 1 else f\'{m:.{digits}f}\'\n    if s is not None and not (isinstance(s, float) and math.isnan(s)):\n        text += \' ± \' + (f\'{s:.{digits}f}\'.replace(\'0.\', \'.\', 1))\n    return text\n\n\ndef _style_axes(ax):\n    for side in [\'top\', \'right\']:\n        ax.spines[side].set_visible(False)\n    for side in [\'left\', \'bottom\']:\n        ax.spines[side].set_color(GRID)\n    ax.tick_params(colors=INK2, labelsize=9)\n    ax.yaxis.grid(True, color=GRID, linewidth=.8)\n    ax.set_axisbelow(True)\n    ax.title.set_color(INK)\n\n\ndef _plt():\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\': 10, \'axes.titlesize\': 11, \'axes.titleweight\': \'bold\',\n                         \'axes.labelcolor\': INK2, \'figure.facecolor\': \'white\'})\n    return plt\n\n\n# ----------------------------------------------------------------- Table 1 / Fig A.1\ndef table1_datasets(df, audit, author_root=None):\n    rows = []\n    for ds in DATASETS:\n        if ds not in set(df.dataset) and ds not in audit:\n            continue\n        a = audit.get(ds, {})\n        sp = a.get(\'splits\', {})\n        raw_steps = \'–\'\n        if author_root:\n            raw, t = RAW[ds]\n            p = Path(author_root) / \'all_data\' / raw / str(t) / f\'ml_{raw}.csv\'\n            if p.exists():\n                raw_steps = int(pd.read_csv(p, usecols=[\'timestamp\']).timestamp.nunique())\n        rows.append({\'Dataset\': LABEL[ds], \'Domain\': DOMAIN[ds],\n                     \'# Nodes\': a.get(\'nodes\'), \'# Edges\': a.get(\'edges\'),\n                     \'Paper # Nodes / # Edges\': \'{:,} / {:,}\'.format(*PAPER_T1[ds]),\n                     \'# Time steps\': raw_steps,\n                     \'Train / Val / Test queries\': \'{} / {} / {}\'.format(\n                         sp.get(\'train\', {}).get(\'queries\', \'–\'), sp.get(\'val\', {}).get(\'queries\', \'–\'),\n                         sp.get(\'test\', {}).get(\'queries\', \'–\')),\n                     \'Train sequences > 512 tokens\': sp.get(\'train\', {}).get(\'truncated\', \'–\'),\n                     \'Test inputs > 1024 tokens\': sp.get(\'test\', {}).get(\'above_context\', \'–\')})\n    return pd.DataFrame(rows)\n\n\ndef fig_temporal_pattern(author_root, out):\n    plt = _plt()\n    present = []\n    for ds in DATASETS:\n        raw, t = RAW[ds]\n        p = Path(author_root) / \'all_data\' / raw / str(t) / f\'ml_{raw}.csv\'\n        if p.exists():\n            present.append((ds, pd.read_csv(p, usecols=[\'timestamp\']).timestamp.value_counts().sort_index()))\n    if not present:\n        return None\n    fig, axes = plt.subplots(1, len(present), figsize=(6.2 * len(present), 3.6), squeeze=False)\n    for ax, (ds, counts) in zip(axes[0], present):\n        last = counts.index.max()\n        colors = [\'#2a78d6\' if i < last - 1 else (\'#eda100\' if i == last - 1 else \'#e34948\') for i in counts.index]\n        ax.bar(counts.index, counts.values, color=colors, width=.8, edgecolor=\'white\', linewidth=1)\n        ax.set_title(f\'{LABEL[ds]}: interactions per time step\')\n        ax.set_xlabel(\'time step\')\n        ax.set_ylabel(\'# interactions\')\n        ax.set_xticks(counts.index)\n        _style_axes(ax)\n        from matplotlib.patches import Patch\n        ax.legend(handles=[Patch(color=\'#2a78d6\', label=\'train\'), Patch(color=\'#eda100\', label=\'validation (T−1)\'),\n                           Patch(color=\'#e34948\', label=\'test (T)\')], frameon=False, fontsize=8)\n    fig.suptitle(\'Figure A.1 (reproduced): temporal pattern of each dataset\', color=INK, fontsize=12)\n    fig.tight_layout()\n    path = Path(out) / \'fig_A1_temporal_pattern.png\'\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n    return path\n\n\n# ----------------------------------------------------------------- trivial baselines\ndef _read(path):\n    return [x.split() for x in Path(path).read_text(encoding=\'utf-8\').splitlines() if x.strip()]\n\n\ndef _targets(line):\n    i = line.index(\'<|pre|>\')\n    j = line.index(\'<|endofpre|>\') if \'<|endofpre|>\' in line else len(line)\n    return [x for x in line[i + 1:j] if not x.startswith(\'<|\')]\n\n\ndef _ndcg(pred, truth):\n    p, y = list(dict.fromkeys(pred)), list(dict.fromkeys(truth))\n    den = sum(1 / math.log2(i + 2) for i in range(min(5, len(y))))\n    return sum(1 / math.log2(i + 2) for i, x in enumerate(p[:5]) if x in y) / den\n\n\ndef _jac(pred, truth):\n    a, b = set(pred), set(truth)\n    return len(a & b) / len(a | b)\n\n\ndef trivial_scores(author_root):\n    rows = []\n    for ds in DATASETS:\n        folder = Path(author_root) / \'resources\' / ds / \'12\'\n        if not (folder / \'test.link_prediction\').exists():\n            continue\n        pop = collections.Counter(x for line in _read(folder / \'train.link_prediction\') for x in _targets(line))\n        ranked = [x for x, _ in pop.most_common(10)]\n        acc = collections.defaultdict(list)\n        for inp, gt in zip(_read(folder / \'test.link_prediction\'), _read(folder / \'test_gt.link_prediction\')):\n            ego = inp[2]\n            truth = [x for x in _targets(gt) if x != ego]\n            if not truth:\n                continue\n            end = inp.index(\'<|endofhistory|>\')\n            slots, cur = [], None\n            for x in inp[3:end]:\n                if x.startswith(\'<|time\'):\n                    if cur:\n                        slots.append(cur)\n                    cur = []\n                elif not x.startswith(\'<|\'):\n                    cur = (cur or []) + [x]\n            if cur:\n                slots.append(cur)\n            for name, pred in [(\'Popular top-1 (train)\', [x for x in ranked if x != ego][:1]),\n                               (\'Repeat last active slot\', slots[-1] if slots else [])]:\n                acc[name].append((_ndcg(pred, truth), _jac(pred, truth)))\n        for name, v in acc.items():\n            a = np.array(v)\n            rows.append(dict(dataset=ds, method=name, ndcg5=a[:, 0].mean(), jaccard=a[:, 1].mean()))\n    return pd.DataFrame(rows)\n\n\n# ----------------------------------------------------------------- Table 2\ndef _agg(g, col):\n    return g[col].mean(), g[col].std(ddof=1)\n\n\ndef table2_performance(df, trivial, ndcg_col=\'ndcg5\'):\n    """Rows: paper methods + trivial; columns per dataset: ours / paper for NDCG@5 and Jaccard.\n    Baseline Jaccard uses max over k=1,5,10,20 on test (paper protocol, oracle); SimpleDyG uses the generated set."""\n    full = df[df.variant == \'full\']\n    rows, numeric = [], []\n    methods = list(PAPER_T2) + (list(trivial.method.unique()) if trivial is not None and len(trivial) else [])\n    for method in methods:\n        row, num = {\'Method\': method}, {\'Method\': method}\n        for ds in DATASETS:\n            if ds not in set(df.dataset):\n                continue\n            g = full[(full.dataset == ds) & (full.model == method)]\n            jcol = \'jaccard\' if method == \'SimpleDyG\' else \'jaccard_oracle_max_k\'\n            if len(g):\n                (nm, ns), (jm, js) = _agg(g, ndcg_col), _agg(g, jcol)\n                n = len(g)\n            elif trivial is not None and method in set(trivial.method):\n                t = trivial[(trivial.dataset == ds) & (trivial.method == method)]\n                nm, ns, jm, js, n = (float(t.ndcg5.iloc[0]), None, float(t.jaccard.iloc[0]), None, 0) if len(t) else (None,) * 5\n            else:\n                nm = ns = jm = js = n = None\n            p = PAPER_T2.get(method, {}).get(ds)\n            row[f\'{LABEL[ds]} NDCG@5 (ours)\'] = _fmt(nm, ns)\n            row[f\'{LABEL[ds]} NDCG@5 (paper)\'] = _fmt(*p[:2]) if p else \'–\'\n            row[f\'{LABEL[ds]} Jaccard (ours)\'] = _fmt(jm, js)\n            row[f\'{LABEL[ds]} Jaccard (paper)\'] = _fmt(*p[2:]) if p else \'–\'\n            num[f\'{LABEL[ds]} NDCG@5 (ours)\'], num[f\'{LABEL[ds]} Jaccard (ours)\'] = nm, jm\n            num[f\'{LABEL[ds]} NDCG@5 (paper)\'] = p[0] if p else None\n            num[f\'{LABEL[ds]} Jaccard (paper)\'] = p[2] if p else None\n        rows.append(row)\n        numeric.append(num)\n    return pd.DataFrame(rows), pd.DataFrame(numeric)\n\n\ndef html_table(text_df, numeric_df=None, caption=\'\', note=\'\'):\n    """HTML table; in every numeric column the best value is bold and the runner-up underlined (paper style)."""\n    marks = {}\n    if numeric_df is not None:\n        for col in numeric_df.columns:\n            if col == \'Method\':\n                continue\n            values = pd.to_numeric(numeric_df[col], errors=\'coerce\')\n            order = values.dropna().sort_values(ascending=False)\n            if len(order) >= 1:\n                marks[(order.index[0], col)] = \'b\'\n            if len(order) >= 2:\n                marks[(order.index[1], col)] = \'u\'\n    head = \'\'.join(f\'<th>{html.escape(str(c))}</th>\' for c in text_df.columns)\n    body = []\n    for i, r in text_df.iterrows():\n        cells = []\n        for c in text_df.columns:\n            v = html.escape(str(r[c]))\n            m = marks.get((i, c))\n            v = f\'<b>{v}</b>\' if m == \'b\' else (f\'<u>{v}</u>\' if m == \'u\' else v)\n            cells.append(f\'<td>{v}</td>\')\n        body.append(\'<tr>\' + \'\'.join(cells) + \'</tr>\')\n    style = (\'<style>.pt{border-collapse:collapse;font:12px/1.4 system-ui,sans-serif;margin:6px 0 2px}\'\n             \'.pt th,.pt td{padding:4px 8px;border-bottom:1px solid #e6e5e1;text-align:right;white-space:nowrap}\'\n             \'.pt th{border-bottom:2px solid #0b0b0b;color:#0b0b0b}.pt td:first-child,.pt th:first-child{text-align:left}\'\n             \'.pt caption{caption-side:top;text-align:left;font-weight:600;padding-bottom:4px}\'\n             \'.ptn{font:11px system-ui,sans-serif;color:#52514e;margin:0 0 14px}</style>\')\n    return (style + f\'<table class="pt"><caption>{html.escape(caption)}</caption><thead><tr>{head}</tr></thead>\'\n            f\'<tbody>{"".join(body)}</tbody></table><p class="ptn">{html.escape(note)}</p>\')\n\n\ndef fig_table2_bars(df, trivial, out, ndcg_col=\'ndcg5\'):\n    plt = _plt()\n    full = df[df.variant == \'full\']\n    present = [d for d in DATASETS if d in set(df.dataset)]\n    fig, axes = plt.subplots(len(present), 2, figsize=(11, 3.4 * len(present)), squeeze=False)\n    models = [m for m in [\'TGAT\', \'GraphMixer\', \'SimpleDyG\'] if m in set(full.model)]\n    for r, ds in enumerate(present):\n        for c, (metric, title) in enumerate([(ndcg_col, \'NDCG@5\'), (\'jaccard\', \'Jaccard\')]):\n            ax = axes[r][c]\n            means, stds, seeds = [], [], []\n            for m in models:\n                g = full[(full.dataset == ds) & (full.model == m)]\n                col = metric if (metric != \'jaccard\' or m == \'SimpleDyG\') else \'jaccard_oracle_max_k\'\n                means.append(g[col].mean()); stds.append(g[col].std(ddof=1)); seeds.append(g[col].to_numpy())\n            x = np.arange(len(models))\n            ax.bar(x, means, width=.6, color=[COLOR[m] for m in models], edgecolor=\'white\', linewidth=2,\n                   yerr=stds, capsize=3, error_kw=dict(ecolor=INK2, lw=1))\n            for i, s in enumerate(seeds):\n                ax.scatter(np.full(len(s), x[i]) + np.linspace(-.18, .18, len(s)), s, s=10, color=INK, alpha=.35, zorder=3)\n                top = max(means[i] + (0 if np.isnan(stds[i]) else stds[i]), s.max() if len(s) else means[i])\n                ax.text(x[i], top, _fmt(means[i]), ha=\'center\', va=\'bottom\', fontsize=8, color=INK, fontweight=\'bold\')\n            paper = [PAPER_T2[m][ds][0 if c == 0 else 2] for m in models]\n            ax.scatter(x + .33, paper, marker=\'D\', s=36, color=\'white\', edgecolor=INK, linewidth=1.4, zorder=4,\n                       label=\'paper (Table 2)\')\n            if trivial is not None and len(trivial):\n                for name, style in [(\'Repeat last active slot\', \'--\'), (\'Popular top-1 (train)\', \':\')]:\n                    t = trivial[(trivial.dataset == ds) & (trivial.method == name)]\n                    if len(t):\n                        ax.axhline(float(t[(\'ndcg5\' if c == 0 else \'jaccard\')].iloc[0]), ls=style, color=NEUTRAL,\n                                   lw=1.5, label=name + \' (no training)\')\n            ax.set_xticks(x, models)\n            ax.set_title(f\'{LABEL[ds]} – {title}\')\n            _style_axes(ax)\n            if r == 0 and c == 1:\n                ax.legend(frameon=False, fontsize=8, loc=\'upper left\')\n    fig.suptitle(\'Table 2 (reproduced): mean ± std over seeds, dots = seeds, ◇ = paper\', color=INK, fontsize=12)\n    fig.tight_layout()\n    path = Path(out) / \'fig_table2_bars.png\'\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n    return path\n\n\n# ----------------------------------------------------------------- Table 4\ndef table4_temporal(df, ndcg_col=\'ndcg5\'):\n    s = df[df.model == \'SimpleDyG\']\n    rows = []\n    for ds in [d for d in DATASETS if d in set(s.dataset)]:\n        for protocol in sorted(set(s[s.dataset == ds].protocol)):\n            g = s[(s.dataset == ds) & (s.protocol == protocol)]\n            base = g[g.variant == \'full\'].set_index(\'seed\')\n            for v in [x for x in VARIANT_ORDER if x in set(g.variant)]:\n                gv = g[g.variant == v]\n                row = {\'Dataset\': LABEL[ds], \'Protocol\': protocol, \'Variant\': VARIANT_LABEL[v], \'n seeds\': gv.seed.nunique(),\n                       \'NDCG@5 (ours)\': _fmt(*_agg(gv, ndcg_col)), \'Jaccard (ours)\': _fmt(*_agg(gv, \'jaccard\'))}\n                p = PAPER_T4.get(v, {}).get(ds)\n                row[\'NDCG@5 (paper)\'] = _fmt(*p[:2]) if p else \'–\'\n                row[\'Jaccard (paper)\'] = _fmt(*p[2:]) if p else \'–\'\n                if v != \'full\' and len(base):\n                    other = gv.set_index(\'seed\')\n                    common = base.index.intersection(other.index)\n                    d = (other.loc[common, ndcg_col] - base.loc[common, ndcg_col]).to_numpy()\n                    if len(d) >= 2:\n                        h = T975.get(len(d) - 1, 1.96) * d.std(ddof=1) / math.sqrt(len(d))\n                        row[\'ΔNDCG vs full [95% CI]\'] = f\'{d.mean():+.4f} [{d.mean() - h:+.4f}, {d.mean() + h:+.4f}]\'\n                        row[\'Significant?\'] = \'yes\' if (d.mean() - h > 0 or d.mean() + h < 0) else \'no\'\n                rows.append(row)\n    return pd.DataFrame(rows).fillna(\'–\')\n\n\ndef fig_table4_ablation(df, out, ndcg_col=\'ndcg5\'):\n    plt = _plt()\n    s = df[df.model == \'SimpleDyG\']\n    groups = [(ds, p) for ds in DATASETS for p in sorted(set(s[s.dataset == ds].protocol))]\n    fig, axes = plt.subplots(len(groups), 2, figsize=(11, 3.4 * len(groups)), squeeze=False)\n    for r, (ds, protocol) in enumerate(groups):\n        g = s[(s.dataset == ds) & (s.protocol == protocol)]\n        variants = [v for v in VARIANT_ORDER if v in set(g.variant)]\n        for c, (metric, title) in enumerate([(ndcg_col, \'NDCG@5\'), (\'jaccard\', \'Jaccard\')]):\n            ax = axes[r][c]\n            x = np.arange(len(variants))\n            for i, v in enumerate(variants):\n                vals = g[g.variant == v][metric].to_numpy()\n                m, sd = vals.mean(), (vals.std(ddof=1) if len(vals) > 1 else 0)\n                ax.bar(i, m, width=.6, color=COLOR[v], edgecolor=\'white\', linewidth=2, yerr=sd, capsize=3,\n                       error_kw=dict(ecolor=INK2, lw=1))\n                ax.scatter(np.full(len(vals), i) + np.linspace(-.18, .18, len(vals)), vals, s=10, color=INK, alpha=.35, zorder=3)\n                ax.text(i, max(m + sd, vals.max()), _fmt(m), ha=\'center\', va=\'bottom\', fontsize=8, color=INK,\n                        fontweight=\'bold\')\n                p = PAPER_T4.get(v, {}).get(ds)\n                if p and protocol == \'author_compat\':\n                    ax.scatter(i + .33, p[0 if c == 0 else 2], marker=\'D\', s=36, color=\'white\', edgecolor=INK,\n                               linewidth=1.4, zorder=4, label=\'paper (Table 4)\' if i == 0 else None)\n            ax.set_xticks(x, [VARIANT_LABEL[v] for v in variants], rotation=12)\n            ax.set_title(f\'{LABEL[ds]} ({protocol}) – {title}\')\n            _style_axes(ax)\n            if c == 1 and protocol == \'author_compat\':\n                ax.legend(frameon=False, fontsize=8)\n    fig.suptitle(\'Table 4 (reproduced): temporal alignment designs, mean ± std, dots = seeds\', color=INK, fontsize=12)\n    fig.tight_layout()\n    path = Path(out) / \'fig_table4_ablation.png\'\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n    return path\n\n\ndef fig_paired_seeds(df, out, ndcg_col=\'ndcg5\'):\n    plt = _plt()\n    s = df[df.model == \'SimpleDyG\']\n    panels = []\n    for ds in DATASETS:\n        for protocol in sorted(set(s[s.dataset == ds].protocol)):\n            g = s[(s.dataset == ds) & (s.protocol == protocol)]\n            if \'full\' in set(g.variant):\n                panels += [(ds, protocol, v) for v in VARIANT_ORDER if v in set(g.variant) and v != \'full\']\n    if not panels:\n        return None\n    cols = min(3, len(panels))\n    rows = math.ceil(len(panels) / cols)\n    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.6 * rows), squeeze=False)\n    for ax in axes.flat[len(panels):]:\n        ax.set_visible(False)\n    for ax, (ds, protocol, v) in zip(axes.flat, panels):\n        g = s[(s.dataset == ds) & (s.protocol == protocol)]\n        base = g[g.variant == \'full\'].set_index(\'seed\')[ndcg_col]\n        o = g[g.variant == v].set_index(\'seed\')[ndcg_col]\n        common = base.index.intersection(o.index)\n        up = int((o[common] > base[common] + 1e-12).sum()); down = int((o[common] < base[common] - 1e-12).sum())\n        for seed in common:\n            better = o[seed] > base[seed]\n            ax.plot([0, 1], [base[seed], o[seed]], color=COLOR[v] if better else NEUTRAL, lw=1.3, alpha=.8)\n        ax.scatter(np.zeros(len(common)), base[common], s=22, color=COLOR[\'full\'], edgecolor=\'white\', zorder=3)\n        ax.scatter(np.ones(len(common)), o[common], s=22, color=COLOR[v], edgecolor=\'white\', zorder=3)\n        ax.set_xticks([0, 1], [\'full\', VARIANT_LABEL[v]])\n        ax.set_xlim(-.3, 1.3)\n        ax.set_ylabel(\'test NDCG@5\')\n        ax.set_title(f\'{LABEL[ds]} ({protocol})\\n{up} seeds up, {down} down (colored = up)\', fontsize=10)\n        _style_axes(ax)\n    fig.suptitle(\'Same seed, full → variant\', color=INK, fontsize=12)\n    fig.tight_layout()\n    path = Path(out) / \'fig_paired_seeds.png\'\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n    return path\n\n\n# ----------------------------------------------------------------- curves / dims / time / collapse\ndef fig_training_curves(df, runs, out):\n    plt = _plt()\n    s = df[df.model == \'SimpleDyG\']\n    groups = [(ds, p) for ds in DATASETS for p in sorted(set(s[s.dataset == ds].protocol))]\n    fig, axes = plt.subplots(1, len(groups), figsize=(5.8 * len(groups), 3.8), squeeze=False)\n    drawn = False\n    for ax, (ds, protocol) in zip(axes[0], groups):\n        g = s[(s.dataset == ds) & (s.protocol == protocol)]\n        key = \'ndcg5_author\' if protocol == \'author_compat\' else \'ndcg5\'\n        for v in [x for x in VARIANT_ORDER if x in set(g.variant)]:\n            curves = []\n            for rid in g[g.variant == v].run_id:\n                p = runs.get(rid, Path(\'_\')) / \'history.json\'\n                if p.exists():\n                    h = json.loads(p.read_text(encoding=\'utf-8\'))\n                    curves.append(pd.Series([e[key] for e in h], index=[e[\'epoch\'] for e in h]))\n            if not curves:\n                continue\n            for cu in curves:\n                ax.plot(cu.index, cu.values, color=COLOR[v], lw=.6, alpha=.25)\n            table = pd.concat(curves, axis=1)\n            alive = table.notna().sum(axis=1)\n            mean = table.mean(axis=1)[alive >= min(3, len(curves))]\n            ax.plot(mean.index, mean.values, color=COLOR[v], lw=2, label=VARIANT_LABEL[v])\n            drawn = True\n        ax.set_xlabel(\'epoch\')\n        ax.set_ylabel(f\'validation {key}\')\n        ax.set_title(f\'{LABEL[ds]} ({protocol}): validation curves\\n(thin = seeds, thick = mean while ≥3 seeds still training)\', fontsize=10)\n        _style_axes(ax)\n        ax.legend(frameon=False, fontsize=8)\n    fig.tight_layout()\n    path = Path(out) / \'fig_validation_curves.png\'\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n    return path if drawn else None\n\n\ndef fig_baseline_dims(df, runs, out):\n    plt = _plt()\n    rows = []\n    for _, r in df[df.model.isin([\'TGAT\', \'GraphMixer\'])].iterrows():\n        p = runs.get(r.run_id, Path(\'_\')) / \'validation_search.json\'\n        if p.exists():\n            for e in json.loads(p.read_text(encoding=\'utf-8\')):\n                rows.append(dict(dataset=r.dataset, model=r.model, seed=r.seed, dim=e[\'dim\'],\n                                 ndcg5=e[\'validation\'][\'ndcg5\'], seconds=e.get(\'training_seconds\'),\n                                 epochs=e.get(\'epochs_run\')))\n    if not rows:\n        return None, pd.DataFrame()\n    d = pd.DataFrame(rows)\n    present = [x for x in DATASETS if x in set(d.dataset)]\n    fig, axes = plt.subplots(1, len(present), figsize=(5.8 * len(present), 3.6), squeeze=False)\n    for ax, ds in zip(axes[0], present):\n        for m in [\'TGAT\', \'GraphMixer\']:\n            g = d[(d.dataset == ds) & (d.model == m)].groupby(\'dim\').ndcg5.agg([\'mean\', \'std\'])\n            if len(g):\n                ax.errorbar(g.index, g[\'mean\'], yerr=g[\'std\'], color=COLOR[m], lw=2, marker=\'o\', ms=6, capsize=3, label=m)\n        ax.set_xscale(\'log\', base=2)\n        ax.set_xticks(sorted(d.dim.unique()), [str(x) for x in sorted(d.dim.unique())])\n        ax.set_xlabel(\'hidden dimension\')\n        ax.set_ylabel(\'validation NDCG@5\')\n        ax.set_title(f\'{LABEL[ds]}: baselines vs hidden dimension\')\n        _style_axes(ax)\n        ax.legend(frameon=False, fontsize=8)\n    fig.suptitle(\'Figure 4(c)-style: hidden dimension (baselines’ validation search; SimpleDyG not swept)\', color=INK, fontsize=11)\n    fig.tight_layout()\n    path = Path(out) / \'fig_baseline_dims.png\'\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n    return path, d\n\n\ndef table5_time(df, dims):\n    rows = []\n    for (ds, model, variant, protocol), g in df.groupby([\'dataset\', \'model\', \'variant\', \'protocol\']):\n        if model == \'SimpleDyG\':\n            per_epoch = (g.training_seconds / g.epochs_run).mean()\n            epochs = g.epochs_run.mean()\n        else:\n            dd = dims[(dims.dataset == ds) & (dims.model == model)] if len(dims) else pd.DataFrame()\n            if len(dd) and dd.seconds.notna().all():\n                per_epoch, epochs = dd.seconds.sum() / dd.epochs.sum(), dd.epochs.sum() / g.seed.nunique()\n            else:\n                per_epoch, epochs = (g.training_seconds / g.epochs_run).mean(), g.epochs_run.mean()\n        rows.append({\'Dataset\': LABEL.get(ds, ds), \'Method\': model if model != \'SimpleDyG\' else f\'SimpleDyG ({variant})\',\n                     \'Protocol\': protocol, \'Seconds / epoch (ours, T4)\': round(per_epoch, 1),\n                     \'Epochs / run\': round(epochs, 1), \'Total train hours / run\': round(g.training_seconds.mean() / 3600, 2),\n                     \'Test seconds / run\': round(g.testing_seconds.mean(), 1),\n                     \'Parameters\': f\'{int(g.parameters.mean()):,}\', \'Peak GPU MB\': round(g.peak_gpu_mb.mean()),\n                     \'Paper s / epoch (UCI, L40)\': PAPER_T5.get(model, \'–\') if ds == \'UCI_13\' and variant == \'full\' else \'–\'})\n    note = (\'Ours includes validation decoding after every epoch; baselines sum all hidden-dimension candidates. \'\n            \'Paper Table 5 measures training only, on one NVIDIA L40.\')\n    return pd.DataFrame(rows), note\n\n\ndef collapse_table(df, runs, threshold=.7):\n    rows = []\n    for _, r in df[df.model == \'SimpleDyG\'].iterrows():\n        p = runs.get(r.run_id, Path(\'_\')) / \'test_predictions.jsonl\'\n        if not p.exists():\n            continue\n        recs = [json.loads(x) for x in p.read_text(encoding=\'utf-8\').splitlines() if x.strip()]\n        recs = [x for x in recs if x.get(\'truth\')]\n        keys = collections.Counter(json.dumps(sorted(set(map(str, x[\'pred\'])))) for x in recs)\n        top, cnt = keys.most_common(1)[0]\n        rows.append(dict(dataset=r.dataset, protocol=r.protocol, variant=r.variant, seed=r.seed,\n                         top_share=cnt / len(recs), top_prediction=top, distinct=len(keys), ndcg5=r.ndcg5,\n                         collapsed=cnt / len(recs) >= threshold))\n    return pd.DataFrame(rows)\n\n\ndef fig_collapse(col, out, threshold=.7):\n    if col.empty:\n        return None\n    plt = _plt()\n    groups = [(ds, p) for ds in DATASETS for p in sorted(set(col[col.dataset == ds].protocol))]\n    fig, ax = plt.subplots(figsize=(max(7, 1.3 * sum(col[(col.dataset == d) & (col.protocol == p)].variant.nunique() for d, p in groups) + 2), 4))\n    ticks = []\n    pos = 0\n    for ds, protocol in groups:\n        g = col[(col.dataset == ds) & (col.protocol == protocol)]\n        for v in [x for x in VARIANT_ORDER if x in set(g.variant)]:\n            vals = g[g.variant == v].top_share.to_numpy()\n            ax.scatter(np.full(len(vals), pos) + np.linspace(-.2, .2, len(vals)), vals, s=26, color=COLOR[v],\n                       edgecolor=\'white\', zorder=3)\n            short = {\'author_compat\': \'compat\', \'corrected\': \'corrected\'}.get(protocol, protocol)\n            ticks.append((pos, f\'{LABEL[ds]}\\n{short}\\n{v.replace("_", " ")}\'))\n            pos += 1\n        pos += .6\n    ax.axhline(threshold, color=\'#e34948\', ls=\'--\', lw=1.2, label=f\'collapse threshold ({threshold:.0%})\')\n    ax.set_xticks([t[0] for t in ticks], [t[1] for t in ticks], fontsize=7)\n    ax.set_ylim(0, 1.05)\n    ax.set_ylabel(\'share of test queries with\\nthe single most common prediction\')\n    ax.set_title(\'Prediction collapse check (each dot = one run)\')\n    _style_axes(ax)\n    ax.legend(frameon=False, fontsize=8, loc=\'center right\')\n    fig.tight_layout()\n    path = Path(out) / \'fig_collapse.png\'\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n    return path\n\n\n# ----------------------------------------------------------------- driver\ndef render_all(result_roots, author_root=None, out_dir=\'paper_figures\', ndcg_col=\'ndcg5\'):\n    out = Path(out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    df, runs = load_results(result_roots)\n    audit = load_audit(result_roots)\n    trivial = trivial_scores(author_root) if author_root else None\n    res = {\'figures\': {}, \'tables\': {}, \'html\': []}\n    metric_note = (\'NDCG@5 = standard (deduplicated) metric\' if ndcg_col == \'ndcg5\'\n                   else \'NDCG@5 = author-compatible metric (keeps duplicates)\')\n\n    t1 = table1_datasets(df, audit, author_root)\n    res[\'tables\'][\'table1_datasets\'] = t1\n    res[\'html\'].append(html_table(t1, caption=\'Table 1 (reproduced): dataset statistics\',\n                                  note=\'Node/edge counts are taken from the data audit of the released preprocessed files.\'))\n    t2, t2n = table2_performance(df, trivial, ndcg_col)\n    res[\'tables\'][\'table2_performance\'] = t2\n    res[\'html\'].append(html_table(t2, t2n.drop(columns=[c for c in t2n.columns if \'paper\' in c]),\n        caption=\'Table 2 (reproduced): dynamic link prediction, mean ± std over seeds (bold = best, underline = runner-up among our runs)\',\n        note=metric_note + \'. Baseline Jaccard = max over k∈{1,5,10,20} on test as in the paper (oracle); \'\n             \'SimpleDyG Jaccard uses its generated set. Six paper baselines were not run (–). \'\n             \'Trivial rows use no training.\'))\n    t4 = table4_temporal(df, ndcg_col)\n    res[\'tables\'][\'table4_temporal\'] = t4\n    res[\'html\'].append(html_table(t4, caption=\'Table 4 (reproduced): impact of temporal alignment designs\',\n        note=metric_note + \'. Δ is paired by seed against full of the same dataset and protocol; \'\n             \'CI is t-based. Paper values apply to the author_compat protocol only.\'))\n    dims_path, dims = fig_baseline_dims(df, runs, out)\n    t5, t5note = table5_time(df, dims)\n    res[\'tables\'][\'table5_time\'] = t5\n    res[\'html\'].append(html_table(t5, caption=\'Table 5 (reproduced): time efficiency\', note=t5note))\n    col = collapse_table(df, runs)\n    if len(col):\n        summary = (col.groupby([\'dataset\', \'protocol\', \'variant\'])\n                   .agg(runs=(\'seed\', \'count\'), collapsed=(\'collapsed\', \'sum\'), top_share_min=(\'top_share\', \'min\'),\n                        top_share_max=(\'top_share\', \'max\'), distinct_sets_max=(\'distinct\', \'max\')).reset_index())\n        summary[\'dataset\'] = summary.dataset.map(LABEL)\n        for c in [\'top_share_min\', \'top_share_max\']:\n            summary[c] = summary[c].map(lambda v: f\'{v:.1%}\')\n        res[\'tables\'][\'collapse_summary\'] = summary\n        res[\'html\'].append(html_table(summary, caption=\'Prediction collapse summary (SimpleDyG, test)\',\n            note=\'A run is collapsed when one prediction set is given to ≥70% of test queries.\'))\n\n    for name, fn in [(\'fig_A1_temporal_pattern\', lambda: fig_temporal_pattern(author_root, out) if author_root else None),\n                     (\'fig_table2_bars\', lambda: fig_table2_bars(df, trivial, out, ndcg_col)),\n                     (\'fig_table4_ablation\', lambda: fig_table4_ablation(df, out, ndcg_col)),\n                     (\'fig_paired_seeds\', lambda: fig_paired_seeds(df, out, ndcg_col)),\n                     (\'fig_validation_curves\', lambda: fig_training_curves(df, runs, out)),\n                     (\'fig_baseline_dims\', lambda: dims_path),\n                     (\'fig_collapse\', lambda: fig_collapse(col, out))]:\n        try:\n            path = fn()\n        except Exception as exc:  # one failing figure must not hide the others\n            print(\'Figure skipped:\', name, \'-\', exc)\n            path = None\n        if path:\n            res[\'figures\'][name] = Path(path)\n    for name, table in res[\'tables\'].items():\n        table.to_csv(out / f\'{name}.csv\', index=False)\n    page = [\'<!DOCTYPE html><html><head><meta charset="utf-8"><title>SimpleDyG – paper-style tables</title></head>\'\n            \'<body style="font-family:system-ui,sans-serif;max-width:1200px;margin:24px auto;padding:0 16px">\',\n            \'<h1 style="font-size:20px">SimpleDyG reproduction – paper-style tables and figures</h1>\']\n    page += res[\'html\']\n    for name, path in res[\'figures\'].items():\n        page.append(f\'<h3 style="font-size:14px">{name}</h3><img src="{path.name}" style="max-width:100%">\')\n    (out / \'paper_tables.html\').write_text(\'\\n\'.join(page) + \'</body></html>\', encoding=\'utf-8\')\n    res[\'df\'], res[\'out\'] = df, out\n    return res\n\n\ndef show(res):\n    """Display everything inline in Jupyter."""\n    from IPython.display import HTML, Image, display\n    for h in res[\'html\']:\n        display(HTML(h))\n    for name, path in res[\'figures\'].items():\n        print(name)\n        display(Image(filename=str(path)))\n'
pfig = types.ModuleType("paper_figures")
exec(compile(PAPER_FIGURES_SOURCE, "paper_figures.py", "exec"), pfig.__dict__)
FIGURES = WORK / "simpledyg_paper_figures"
if FIGURES.exists():
    shutil.rmtree(FIGURES)
figures = pfig.render_all(RESULT_ROOTS, AUTHOR_ROOT, FIGURES, ndcg_col=NDCG_METRIC)
df = figures["df"]
print("Tổng số run:", len(df))
display(df.groupby(["dataset", "model", "variant", "protocol"]).seed.nunique().rename("n_seeds").reset_index())


## 4. Bảng (Table 1, 2, 4, 5, suy biến) và biểu đồ

In [ ]:
pfig.show(figures)


## 5. Tải về

In [ ]:
archive = shutil.make_archive(str(WORK / "simpledyg_paper_figures"), "zip", FIGURES)
print("Đã lưu:", archive)
print("Gồm:", sorted(p.name for p in FIGURES.iterdir()))
try:
    from IPython.display import FileLink
    display(FileLink(os.path.relpath(archive)))
except Exception:
    pass


## Gợi ý đọc

- **Table 2:** số của chúng tôi dùng metric chọn ở `NDCG_METRIC`. Jaccard của baseline là max theo k ∈ {1, 5, 10, 20} trên test (như paper, mang tính oracle); Jaccard của SimpleDyG lấy theo tập sinh ra. Sáu baseline paper không chạy được hiển thị “–”.
- **Nét đứt và chấm** là baseline không huấn luyện: lặp lại slot gần nhất có hoạt động, và đích phổ biến nhất trong train. Trên ML-10M, mọi mô hình đều chạm mức “phổ biến” (xem biểu đồ suy biến).
- **Table 4:** cột Δ so cặp cùng seed với `full` cùng dataset/protocol; “Significant?” = CI 95% không chứa 0.
- **Table 5:** giây/epoch trên T4 gồm cả giải mã validation; paper đo trên L40, chỉ phần huấn luyện. Chỉ nên so tương đối.
- Đổi `NDCG_METRIC = "ndcg5_author"` để có số đúng cách tính của paper (khác chủ yếu trên UCI).
